# Fase 3 — Preprocesamiento y features

**TP1 — Aprendizaje Automático (72.75) — ITBA** · Consignas **1.1** y **1.4**

Paso **6 (Feature engineering / selection)** del pipeline de la Clase 3.

Decidimos tres cosas: cómo convertir las categóricas a números, si escalamos y con qué
método, y qué features incluimos. Todo queda en un objeto que se ajusta **sólo con train**.

**El test no se abre en este notebook.**

In [1]:
# ---------------------------------------------------------------------------
# Configuracion del entorno
# ---------------------------------------------------------------------------
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_regression

from src.config import TARGET, RANDOM_SEED
from src.data import cargar_splits

# Funciones propias, documentadas en src/preprocesamiento.py
from src.preprocesamiento import (
    NUMERICAS,                 # ["age", "bmi", "children"]
    CATEGORICAS,               # ["sex", "smoker", "region"]
    separar_X_y,               # divide un conjunto en X e y
    construir_preprocesador,   # arma el ColumnTransformer
    nombres_de_features,       # nombres de las columnas de salida
)

pd.set_option("display.width", 120)

# Solo cargamos el train. El test queda sin abrir hasta la Fase 6.
train, _ = cargar_splits()
X_train, y_train = separar_X_y(train)

print(f"Train: {X_train.shape[0]} filas | {X_train.shape[1]} variables de entrada")

Train: 1069 filas | 6 variables de entrada


---

# 1. Variables categóricas *(consigna 1.1)*

In [2]:
# Cardinalidad y frecuencia de cada nivel.
for col in CATEGORICAS:
    frec = (100 * train[col].value_counts(normalize=True)).round(1)
    print(f"{col:8s} - {train[col].nunique()} niveles: " +
          ", ".join(f"{nivel}={pct}%" for nivel, pct in frec.items()))

sex      - 2 niveles: male=51.4%, female=48.6%
smoker   - 2 niveles: no=80.0%, yes=20.0%
region   - 4 niveles: southeast=26.8%, southwest=24.8%, northwest=24.7%, northeast=23.7%


Las tres son **nominales**: etiquetas sin orden natural. Nada dice que `northeast` valga
más que `southwest`.

| Estrategia (Clase 2) | ¿La usamos? | Motivo |
|---|---|---|
| Ordinal | No | Sólo sirve con jerarquía real (slide 14). Codificar `northeast=1 … southwest=4` haría que el modelo leyera southwest como 4× northeast |
| **One-hot** | Sí | Ver 1.2 |
| Frequency | No | Ver evidencia abajo |
| Target | No | Usa `y` para construir features: habría que recalcularlo dentro de cada fold o el error de validación sale optimista. Y no resuelve nada: la región con menos datos tiene 253 observaciones |
| Regularized target | No | Existe para amortiguar categorías con pocos datos. Sin ese problema, sólo agrega un hiperparámetro (λ) más |

In [3]:
# Frequency encoding: cada categoria se reemplazaria por su frecuencia relativa.
frecuencias = train["region"].value_counts(normalize=True).round(4)
print("Valor que tomaria cada region con frequency encoding:")
print(frecuencias.to_string())
print(f"\nAmplitud total          : {frecuencias.max() - frecuencias.min():.4f}")
print(f"southwest vs northwest  : {abs(frecuencias['southwest'] - frecuencias['northwest']):.4f}")

Valor que tomaria cada region con frequency encoding:
region
southeast    0.2685
southwest    0.2479
northwest    0.2470
northeast    0.2367

Amplitud total          : 0.0318
southwest vs northwest  : 0.0009


**Frequency encoding descartado, con evidencia.** Las cuatro regiones están casi
perfectamente balanceadas (0.2367 a 0.2685). `southwest` y `northwest` quedarían separadas
por **0.0009**: el modelo no podría distinguirlas. El método sirve cuando hay categorías
raras que se comportan distinto; acá no hay ninguna.

## 1.2 Decisión: one-hot con `drop="first"`

- Respeta la naturaleza nominal: no inventa orden.
- No mira el target, así que no hay leakage posible.
- El costo en dimensionalidad es mínimo. La Clase 2 (slide 18) advierte que one-hot
  *"aumenta la dimensionalidad mucho"* pero que *"está bien si no hay muchas variables
  categóricas o muchas categorías"*: tenemos 3 variables con 2, 2 y 4 niveles.

**`drop="first"`** evita la *trampa de las variables dummy*: con las 4 columnas de `region`
su suma sería siempre 1, igual que el intercepto, y los coeficientes dejarían de ser únicos.
La categoría descartada queda como referencia.

**6 variables → 8 columnas.**

| Original | Columnas generadas | Referencia |
|---|---|---|
| `sex` | `sex_male` | `female` |
| `smoker` | `smoker_yes` | `no` |
| `region` | `region_northwest`, `region_southeast`, `region_southwest` | `northeast` |

---

# 2. Escalado *(consigna 1.4)*

In [4]:
# Escalas con las que trabajamos, sin transformar.
print(train[NUMERICAS].agg(["min", "max", "mean", "std"]).T.round(2).to_string())

            min    max   mean    std
age       18.00  64.00  39.20  14.00
bmi       15.96  53.13  30.54   6.05
children   0.00   5.00   1.08   1.19


`age` tiene desvío 14.0 y `children` 1.19: **doce veces de diferencia**.

**Para la regresión lineal por mínimos cuadrados el escalado no cambia nada**: los
coeficientes absorben las unidades y la predicción es idéntica.

**Sí importa con regularización L1** (Fase 5). La penalización es λ·Σ|w| y castiga
coeficientes grandes, pero un coeficiente es "grande" según las unidades de su variable. Sin
escalar, la penalización castigaría más a unas variables que a otras por su escala, no por
su utilidad.

**Elegimos z-score (`StandardScaler`).** La Clase 3 (slide 43) lo da como opción por defecto
y señala que es *"mucho menos sensible a outliers"* que min-max. Además min-max se define
enteramente por el mínimo y el máximo, así que quedaría determinado por los valores extremos
que en la Fase 2 decidimos **conservar**: usarlo sería contradictorio.

**Las columnas one-hot no se escalan:** ya están en 0/1, con desvíos de 0.40 a 0.50, del
mismo orden que una variable estandarizada.

---

# 3. Selección de features *(consigna 1.4)*

Aplicamos **filtros**, la primera familia de la Clase 3 (slide 63): estadísticos
univariados, independientes del modelo. Los usamos como **diagnóstico**, no como decisión
automática, porque no ven interacciones entre variables.

Usamos dos porque miden cosas distintas: **Pearson** capta relación lineal (slide 65) e
**información mutua** capta relaciones de cualquier tipo (slide 66).

In [5]:
# Aplicamos el preprocesador para obtener las 8 columnas finales.
# Se ajusta solo con train, el unico conjunto que podemos mirar.
preprocesador = construir_preprocesador()
X_proc = pd.DataFrame(
    preprocesador.fit_transform(X_train),
    columns=nombres_de_features(preprocesador),
    index=X_train.index,
)

# Pearson: correlacion lineal con el target, en valor absoluto.
pearson = X_proc.corrwith(y_train).abs()

# Informacion mutua. random_state fija el resultado porque el estimador usa
# vecinos mas cercanos con ruido aleatorio.
info_mutua = pd.Series(
    mutual_info_regression(X_proc, y_train, random_state=RANDOM_SEED),
    index=X_proc.columns,
)

filtros = pd.DataFrame({"pearson_abs": pearson.round(3), "info_mutua": info_mutua.round(3)})
filtros["rank_pearson"] = filtros["pearson_abs"].rank(ascending=False).astype(int)
filtros["rank_MI"] = filtros["info_mutua"].rank(ascending=False).astype(int)
filtros.sort_values("info_mutua", ascending=False)

,pearson_abs,info_mutua,rank_pearson,rank_MI
age,0.290,1.444,2,1
smoker_yes,0.774,0.351,1,2
children,0.086,0.158,4,3
sex_male,0.060,0.137,6,4
bmi,0.178,0.075,3,5
region_northwest,0.055,0.042,7,6
region_southeast,0.079,0.028,5,7
region_southwest,0.036,0.014,8,8


**Codificada, `smoker` domina:** `smoker_yes` tiene Pearson **0.774**, cuatro veces más
que cualquier otra. En la Fase 2 Pearson no la veía porque era una columna de texto; una vez
convertida a 0/1 la detecta sin problema.

**Los dos filtros no coinciden y está bien:** Pearson pone primero a `smoker_yes`,
información mutua a `age`. Miden cosas distintas y sus valores no son comparables entre sí.
Lo relevante es que ambos ubican a `smoker_yes` y `age` arriba y a las regiones abajo.

**Ninguna feature da cero.**

In [6]:
# Correlacion absoluta entre features, ignorando la diagonal.
corr_features = X_proc.corr().abs()
sin_diagonal = corr_features.where(~np.eye(len(corr_features), dtype=bool))
var_a, var_b = sin_diagonal.stack().idxmax()
print(f"Correlacion absoluta maxima entre features: {sin_diagonal.max().max():.3f} "
      f"({var_a} - {var_b})")

Correlacion absoluta maxima entre features: 0.348 (region_southeast - region_southwest)


El máximo (0.348) es entre dos dummies de `region`, que están anticorrelacionadas por
construcción: si una vale 1, las otras valen 0. **No es redundancia informativa.**

## Decisión: conservamos las 6 variables

1. Ninguna es irrelevante: ambos filtros dan valores mayores que cero para todas.
2. Ninguna es redundante.
3. No hay maldición de la dimensionalidad: 8 columnas y 1069 filas. La "regla del 10×"
   (Clase 2, slide 91) pediría unas 80 observaciones.

> Donde **sí** va a hacer falta seleccionar es en la Fase 5: la transformación polinómica de
> grado 2 lleva estas 8 columnas a unas 45, y la de grado 3 a más de 160. Ahí entran las
> otras dos familias de la Clase 3, **wrapper** (RFE) y **embedded** (Lasso L1).

---

# 4. El preprocesador

Todo lo decidido se encapsula en un `ColumnTransformer`, definido en
`src/preprocesamiento.py`. Se ajusta con `fit` sobre datos de entrenamiento y se aplica con
`transform` al resto, siempre con los parámetros aprendidos.

En la Fase 4 entra dentro de un `Pipeline`, de modo que la validación cruzada lo reajusta
automáticamente en cada fold.

---

## Conclusiones de la Fase 3

| Decisión | Valor | Motivo |
|---|---|---|
| Encoding | **One-hot con `drop="first"`** | Son nominales; sin leakage; costo mínimo en dimensionalidad |
| Ordinal | Descartado | No hay jerarquía entre categorías |
| Frequency | Descartado | Las 4 regiones tienen frecuencias casi iguales (0.2367–0.2685) |
| Target / Regularized target | Descartados | Riesgo de leakage y ningún problema de cardinalidad (mín. 253 obs.) |
| Escalado | **z-score sobre las numéricas** | Clase 3 slide 43; necesario para que L1 penalice parejo |
| Columnas one-hot | Sin escalar | Ya están en 0/1 |
| Features | **Las 6** → 8 columnas | Ninguna irrelevante ni redundante; sin problema de dimensionalidad |

**Siguiente:** Fase 4 — regresión lineal con k-fold sobre el train (consignas 2.2 y 2.3).